# Flight Delay Predictor — Streamlit App

This notebook is a **read-through / learning copy** of `app.py`.

`app.py` is a [Streamlit](https://streamlit.io) app, meant to be run with:

```
streamlit run app.py
```

Streamlit re-runs the *entire script top-to-bottom* every time a user interacts
with a widget (dropdown, slider, button, etc.). Running the cells below in
Jupyter will execute the logic once, but widgets like `st.selectbox` won't be
interactive outside of an actual Streamlit server — use this notebook to
understand *what each part does*, and use `streamlit run app.py` to actually
use the app.


## 1. Imports

- `streamlit` — the UI framework that turns this script into a web app.
- `pandas` / `numpy` — data handling.
- `joblib` — loads the trained model artifacts saved by `pipeline.py`.
- `category_encoders` — needed so `joblib.load` can unpickle the saved
  `BinaryEncoder` object.


In [ ]:
import streamlit as st

import pandas as pd
import numpy as np
import joblib
import category_encoders as ce


## 2. Page configuration

`st.set_page_config` must be the **first** Streamlit call in the script.
`layout="centered"` keeps the content in a readable, non-full-width column.


In [ ]:
st.set_page_config(
    page_title="Flight Delay Predictor",
    page_icon="✈️",
    layout="centered"
)


## 3. Load the trained model (cached)

`@st.cache_resource` tells Streamlit to run `load_model()` only **once** and
reuse the result on every re-run. Without this, the ~80MB model file would be
reloaded from disk on every single user interaction, which would be slow.

The `.pkl` file bundles everything the app needs at inference time:
- `pipeline` — the fitted KNNImputer + StandardScaler + LogisticRegression chain.
- `binary_encoder` — the fitted BinaryEncoder for ORIGIN/DEST.
- `feature_names` — the exact ordered list of columns the model expects.
- `ohe_cols` / `binary_cols` — which columns need which encoding.


In [ ]:
@st.cache_resource
def load_model():
    artifacts = joblib.load('flight_delay_model.pkl')
    return (
        artifacts['pipeline'],
        artifacts['binary_encoder'],
        artifacts['feature_names'],
        artifacts['ohe_cols'],
        artifacts['binary_cols']
    )

pipeline, binary_encoder, feature_names, ohe_cols, binary_cols = load_model()


## 4. App header

Title and short description shown at the top of the page, followed by a
horizontal divider line.


In [ ]:
st.title("✈️ Flight Delay Predictor")
st.markdown("""
Predict whether your flight will arrive **more than 15 minutes late**
based on flight details and historical patterns.
""")

st.divider()


## 5. User input form

Two side-by-side columns (`st.columns(2)`) collect all the raw flight details
needed to build a single-row DataFrame for prediction:

- **Left column:** month, day of week, carrier, origin airport.
- **Right column:** destination airport, scheduled departure hour, departure
  delay, distance, taxi-out time, expected air time.

`format_func` in the `selectbox` calls converts the numeric option (e.g. `1`)
into a readable label (e.g. `'Jan'`) purely for display — the underlying value
stored in the variable is still the number.


In [ ]:
col1, col2 = st.columns(2)

with col1:
    month = st.selectbox("Month", list(range(1, 13)),
        format_func=lambda x: ['Jan','Feb','Mar','Apr','May','Jun',
                                'Jul','Aug','Sep','Oct','Nov','Dec'][x-1])

    day_of_week = st.selectbox("Day of Week", list(range(1, 8)),
        format_func=lambda x: ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][x-1])

    carrier = st.selectbox("Carrier", [
        'AA', 'AS', 'B6', 'DL', 'EV', 'F9', 'HA', 'MQ', 'NK', 'OO',
        'UA', 'US', 'VX', 'WN'
    ])

    origin = st.text_input("Origin Airport (IATA)", value="JFK").upper()

with col2:
    dest = st.text_input("Destination Airport (IATA)", value="LAX").upper()

    dep_hour = st.slider("Scheduled Departure Hour", 0, 23, 8)

    dep_delay = st.number_input("Departure Delay (minutes)", -30, 300, 0)

    distance = st.number_input("Distance (miles)", 50, 5000, 800)

    taxi_out = st.number_input("Taxi-Out Time (minutes)", 1, 120, 15)

    air_time = st.number_input("Expected Air Time (minutes)", 20, 700, 150)

st.divider()


## 6. Prediction logic

Everything below only runs when the user clicks the **"Predict Delay"**
button (`if st.button(...)`). Because this is a single Streamlit control-flow
block, it's kept as one cell — but it's broken into clearly labelled steps:

1. **Assemble input** — wrap the widget values into a single-row DataFrame
   with the exact column names the model was trained on.
2. **Encode** — apply the *same* BinaryEncoder + one-hot encoding used during
   training (`transform`, not `fit_transform`, so we reuse the training
   mapping instead of learning a new one).
3. **Align columns** — `reindex` fills in any one-hot columns missing from
   this single row (e.g. a carrier that isn't the dropped baseline category)
   with 0, so the columns match `feature_names` exactly.
4. **Predict** — get both the probability of delay and the hard 0/1 label.
5. **Display** — show the probability, a colored success/error message, the
   raw input for verification, and a plain-English confidence note.


In [ ]:
if st.button("Predict Delay", type="primary"):

    # 1. Assemble input as a single-row DataFrame
    input_dict = {
        'MONTH': [month],
        'DAY_OF_WEEK': [day_of_week],
        'OP_UNIQUE_CARRIER': [carrier],
        'ORIGIN': [origin],
        'DEST': [dest],
        'DEP_HOUR': [dep_hour],
        'DEP_DELAY': [dep_delay],
        'TAXI_OUT': [taxi_out],
        'AIR_TIME': [air_time],
        'DISTANCE': [distance]
    }
    input_df = pd.DataFrame(input_dict)

    # 2. Encode using the fitted encoders from training
    input_enc = binary_encoder.transform(input_df)

    for col in ['MONTH', 'DAY_OF_WEEK']:
        input_enc[col] = input_enc[col].astype(str)
    input_enc = pd.get_dummies(input_enc, columns=ohe_cols, drop_first=True)

    # 3. Align columns to exactly match training-time feature order
    input_enc = input_enc.reindex(columns=feature_names, fill_value=0)

    # 4. Run the model
    delay_prob = pipeline.predict_proba(input_enc)[0][1]
    prediction = pipeline.predict(input_enc)[0]

    # 5. Display the result
    st.subheader("Prediction Result")

    st.metric(
        label="Delay Probability",
        value=f"{delay_prob:.1%}"
    )

    st.progress(float(delay_prob))

    if prediction == 1:
        st.error(f"⚠️ This flight is likely to be DELAYED (>{15} min late)")
    else:
        st.success("✅ This flight is likely to arrive ON TIME")

    with st.expander("See input summary"):
        st.dataframe(input_df)

    st.caption(
        f"Model confidence: {'High' if abs(delay_prob - 0.5) > 0.25 else 'Moderate'}. "
        "Based on 2015 BTS domestic flight data. Results are probabilistic estimates."
    )


## 7. Sidebar — model info

A static panel describing the algorithm, imputation strategy, encoding
choices, and prediction target — shown in the Streamlit sidebar for context.


In [ ]:
with st.sidebar:
    st.header("Model Information")
    st.markdown("""
    **Algorithm:** Logistic Regression

    **Imputation:** KNN Imputer (k=5)

    **Encoding:**
    - Binary: ORIGIN, DEST
    - One-Hot: CARRIER, MONTH, DAY_OF_WEEK

    **Target:** ARR_DELAY > 15 minutes

    **Data:** BTS 2015 US Domestic Flights
    """)
    st.divider()
    st.markdown("Built for EPITA MSc Data Science & Analytics")
